# Proyecto Tutorial de Clasficador de Imagenes

# Paso 1: Carga del conjunto de datos

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, GlobalAveragePooling2D, Flatten


In [ ]:
train_dir = Path(r"C:\Users\oswal\Downloads\dogs-vs-cats\dogs-vs-cats\train")

print("¿Existe la carpeta train?:", train_dir.exists())

image_files = sorted(train_dir.glob("*.jpg"))
print("Cantidad total de imágenes:", len(image_files))

In [ ]:
df = pd.DataFrame({
    "filepath": [str(p) for p in image_files],
    "label": [1 if p.name.startswith("dog") else 0 for p in image_files]
})

print(df.head())
print("\nDimensión del dataset:", df.shape)
print("\nDistribución de clases:")
print(df["label"].value_counts())

### Asignamos:
- cat = 0
- dog = 1

# Paso 2: Visualiza la información de entrada

In [ ]:
dog_files = sorted(train_dir.glob("dog.*.jpg"))
cat_files = sorted(train_dir.glob("cat.*.jpg"))

print("Cantidad de perros:", len(dog_files))
print("Cantidad de gatos:", len(cat_files))

## Visualizamos 9 perros

In [ ]:
plt.figure(figsize=(10, 10))

for i, img_path in enumerate(dog_files[:9]):
    img = Image.open(img_path)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.suptitle("Primeras 9 imágenes de perros", fontsize=14)
plt.tight_layout()
plt.show()

## Visualizamos 9 gatos

In [ ]:
plt.figure(figsize=(10, 10))

for i, img_path in enumerate(cat_files[:9]):
    img = Image.open(img_path)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.suptitle("Primeras 9 imágenes de gatos", fontsize=14)
plt.tight_layout()
plt.show()

### Visualizamos las dimensiones de algunas imagenes

In [ ]:
for img_path in dog_files[:5]:
    img = Image.open(img_path)
    print(f"{img_path.name} -> tamaño: {img.size}")

for img_path in cat_files[:5]:
    img = Image.open(img_path)
    print(f"{img_path.name} -> tamaño: {img.size}")

## Redimensionamos las imagenes a un formato de 200x200

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

### Definir parámetros del pipeline

In [ ]:
IMG_SIZE = (200, 200)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

- Todas las imágenes pasarán a 200x200
- Se procesarán en lotes de 32
- TensorFlow optimizará automáticamente parte del flujo

### Carga y preprocesamiento de imágenes

In [ ]:
def load_and_preprocess_image(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label"].values)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["filepath"].values, test_df["label"].values)
)

In [ ]:
train_ds = (
    train_ds
    .shuffle(buffer_size=len(train_df), seed=42)
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
for images, labels in train_ds.take(1):
    print("Shape de imágenes:", images.shape)
    print("Shape de etiquetas:", labels.shape)
    print("Tipo de dato:", images.dtype)
    print("Rango mínimo:", tf.reduce_min(images).numpy())
    print("Rango máximo:", tf.reduce_max(images).numpy())

# Paso 3: Construye una RNA

In [ ]:
model = Sequential()

# Bloque 1
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding="same",
                 activation="relu", input_shape=(200, 200, 3)))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))

# Bloque 2
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))

# Bloque 3
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))

# Bloque 4
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))

# Bloque 5
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same",
                 activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))

# Clasificación
model.add(Flatten())
model.add(Dense(units=4096, activation="relu"))
model.add(Dense(units=4096, activation="relu"))
model.add(Dense(units=2, activation="softmax"))

model.summary()

In [ ]:
del model

In [ ]:
import gc
gc.collect()

### Observación sobre la arquitectura propuesta

Se construyó primero una CNN siguiendo la arquitectura de referencia del módulo, inspirada en VGG, adaptada al tamaño de entrada de `200x200x3`. Esta versión permite mantener coherencia con la propuesta académica del ejercicio y comprobar cómo se organizan sus bloques convolucionales y capas densas.

Sin embargo, al revisar el resumen del modelo, se observa que la red supera los **107 millones de parámetros**, lo que implica un consumo muy alto de memoria y un costo computacional elevado durante el entrenamiento. Esto puede provocar tiempos excesivos de ejecución, riesgo de sobreajuste y un uso poco eficiente de los recursos del computador.

Por esta razón, a continuación se construirá una **versión optimizada de la CNN**, manteniendo la lógica general del modelo visto en el módulo, pero reduciendo su complejidad para hacer el entrenamiento más estable, realista y adecuado para un entorno local.

In [ ]:
model_opt = Sequential()

# Bloque 1
model_opt.add(Conv2D(32, (3, 3), padding="same", activation="relu", input_shape=(200, 200, 3)))
model_opt.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model_opt.add(MaxPool2D(pool_size=(2, 2)))

# Bloque 2
model_opt.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model_opt.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model_opt.add(MaxPool2D(pool_size=(2, 2)))

# Bloque 3
model_opt.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model_opt.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model_opt.add(MaxPool2D(pool_size=(2, 2)))

# Bloque 4
model_opt.add(Conv2D(256, (3, 3), padding="same", activation="relu"))
model_opt.add(Conv2D(256, (3, 3), padding="same", activation="relu"))
model_opt.add(MaxPool2D(pool_size=(2, 2)))

# Clasificación
model_opt.add(GlobalAveragePooling2D())
model_opt.add(Dense(128, activation="relu"))
model_opt.add(Dropout(0.3))
model_opt.add(Dense(2, activation="softmax"))

model_opt.summary()

La versión optimizada del modelo mantiene la estructura general de una CNN para clasificación de imágenes, pero reduce drásticamente la cantidad de parámetros, pasando de más de 107 millones a poco más de 1.2 millones. Esto la vuelve mucho más adecuada para entrenar en un entorno local sin perder coherencia con el objetivo del ejercicio.

In [ ]:
model_opt.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Modelo compilado correctamente")